# 03 - Diffusion Process Visualization

Visualize the forward and reverse diffusion processes for quantum state tomography.

**Forward process**: $q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} x_0, (1-\bar{\alpha}_t)I)$

**Reverse process**: $p_\theta(x_{t-1} | x_t, m) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t, m), \sigma_t^2 I)$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.models.noise_schedules import make_beta_schedule, compute_diffusion_parameters
from src.models.diffusion import DDPM
from src.representation.cholesky import cholesky_to_dm, dm_to_cholesky
from src.data.states import haar_random_pure
from src.evaluation.metrics import fidelity, purity

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)

## 1. Noise Schedules: Linear vs Cosine

In [ ]:
T = 1000

betas_linear = make_beta_schedule(T, 'linear', 1e-4, 0.02).numpy()
betas_cosine = make_beta_schedule(T, 'cosine').numpy()

params_linear = compute_diffusion_parameters(torch.from_numpy(betas_linear))
params_cosine = compute_diffusion_parameters(torch.from_numpy(betas_cosine))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Beta schedule
axes[0].plot(betas_linear, label='Linear')
axes[0].plot(betas_cosine, label='Cosine')
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('beta_t')
axes[0].set_title('Noise Schedule (beta_t)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Alpha bar (cumulative signal retention)
axes[1].plot(params_linear['alphas_cumprod'].numpy(), label='Linear')
axes[1].plot(params_cosine['alphas_cumprod'].numpy(), label='Cosine')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('alpha_bar_t')
axes[1].set_title('Signal Retention (alpha_bar_t)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# SNR = alpha_bar / (1 - alpha_bar)
snr_linear = params_linear['alphas_cumprod'].numpy() / (1 - params_linear['alphas_cumprod'].numpy())
snr_cosine = params_cosine['alphas_cumprod'].numpy() / (1 - params_cosine['alphas_cumprod'].numpy())
axes[2].semilogy(snr_linear, label='Linear')
axes[2].semilogy(snr_cosine, label='Cosine')
axes[2].set_xlabel('Timestep t')
axes[2].set_ylabel('SNR')
axes[2].set_title('Signal-to-Noise Ratio')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Diffusion Noise Schedule Comparison', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Cosine schedule: beta starts at {betas_cosine[0]:.6f}, ends at {betas_cosine[-1]:.6f}')
print(f'  alpha_bar at T/2: {params_cosine["alphas_cumprod"][T//2]:.4f}')
print(f'  alpha_bar at T:   {params_cosine["alphas_cumprod"][-1]:.6f}')

## 2. Forward Diffusion: Visualizing State Destruction

In [ ]:
n_qubits = 2
d = 2**n_qubits

# Create a target state
rho_0 = haar_random_pure(n_qubits, seed=42)
x_0 = torch.from_numpy(dm_to_cholesky(rho_0)).float().unsqueeze(0)

# Set up diffusion parameters (cosine schedule)
betas = make_beta_schedule(1000, 'cosine')
params = compute_diffusion_parameters(betas)

def q_sample(x_0, t):
    noise = torch.randn_like(x_0)
    a_bar = params['sqrt_alphas_cumprod'][t]
    b_bar = params['sqrt_one_minus_alphas_cumprod'][t]
    return a_bar * x_0 + b_bar * noise, noise

# Sample at different timesteps
timesteps = [0, 10, 50, 100, 200, 400, 600, 800, 999]

fig, axes = plt.subplots(2, len(timesteps), figsize=(18, 5))

for i, t in enumerate(timesteps):
    with torch.no_grad():
        x_t, _ = q_sample(x_0, t)
    rho_t = cholesky_to_dm(x_t.numpy()[0])
    fid = fidelity(rho_0, rho_t)
    
    axes[0, i].imshow(np.real(rho_t), cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[0, i].set_title(f't={t}\nFid={fid:.3f}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(np.imag(rho_t), cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[1, i].axis('off')

axes[0, 0].imshow(np.real(rho_0), cmap='RdBu_r', vmin=-0.5, vmax=0.5)
axes[0, 0].set_title(f't=0\n(Original)')

plt.suptitle('Forward Diffusion: Density Matrix Destruction', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Fidelity Decay During Forward Process

In [ ]:
n_qubits = 2
d = 2**n_qubits

rho_0 = haar_random_pure(n_qubits, seed=42)
x_0 = torch.from_numpy(dm_to_cholesky(rho_0)).float()

ts = np.linspace(0, 999, 100).astype(int)
fidelities = []
purities = []

for t in ts:
    x_t, _ = q_sample(x_0.unsqueeze(0), t)
    rho_t = cholesky_to_dm(x_t.numpy()[0])
    fidelities.append(fidelity(rho_0, rho_t))
    purities.append(purity(rho_t))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ts, fidelities)
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('Fidelity to original')
axes[0].set_title('State Fidelity Decay')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ts, purities)
axes[1].axhline(y=1/d, color='red', linestyle='--', label=f'Maximally mixed (1/d={1/d:.3f})')
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('Purity')
axes[1].set_title('Purity During Diffusion')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Forward Diffusion Analysis (n={n_qubits}, cosine schedule)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Reverse Process: How Conditioning Guides Denoising

The reverse process uses the measurement data to guide denoising toward the correct state.
The UNet predicts the noise $\epsilon_\theta(x_t, t, m)$ which is subtracted to recover $x_0$.

In [ ]:
# Schematic of reverse process with conditioning
fig, ax = plt.subplots(figsize=(10, 4))

# Draw the process schematically
stages = [
    ('x_T ~ N(0, I)', 'Pure noise', 'red'),
    ('x_{T-1}', '+ measurement\nconditioning', 'orange'),
    ('...', 'iterative\ndenoising', 'yellow'),
    ('x_1', 'nearly\nreconstructed', 'lightgreen'),
    ('x_0', 'Reconstructed\nCholesky vector', 'green'),
    ('rho = LL^dagger/Tr', 'Valid density\nmatrix!', 'blue'),
]

for i, (label, desc, color) in enumerate(stages):
    ax.add_patch(plt.Rectangle((i*1.5, 0), 1.2, 1.5, 
                                facecolor=color, alpha=0.6, edgecolor='black'))
    ax.text(i*1.5 + 0.6, 0.75, label, ha='center', fontsize=9, fontweight='bold')
    ax.text(i*1.5 + 0.6, 0.3, desc, ha='center', fontsize=8)
    if i < len(stages) - 1:
        ax.arrow(i*1.5 + 1.2, 0.75, 0.2, 0, head_width=0.1, head_length=0.08, fc='black')

ax.set_xlim(-0.2, len(stages)*1.5)
ax.set_ylim(-0.1, 1.8)
ax.axis('off')
ax.set_title('DDPM Reverse Process with Measurement Conditioning', fontsize=14, pad=20)
plt.show()

## Key Insights

- **Cosine schedule**: Preserves signal longer at early timesteps (better for learning fine details)
- **Forward process**: By t=T, states are indistinguishable from pure noise in Cholesky space
- **Reverse process**: The UNet learns to predict and remove noise, guided by measurement data
- **DDIM acceleration**: The same trained model can sample in O(50) steps instead of O(1000)